# 03_gnn_text_v3 — Paper-Backed SOTA Traffic GNN

Upgrades over v2:
- **Text-Conditioned Adjacency** — event text modifies graph connectivity (CrossTrafficLLM)
- **Entity-Masked Cross-Attention** — only roads mentioned in text attend (CadST, T3)
- **Multi-Window Text** — encode t, t-5, t-10 for event timing (T3)
- **Road Metadata** — roadclass, length as static node features (BjTT, Context-GCN)
- **Weighted MSE** — h5×3, h10×2, h15×1 (LLM-ETF)

## Section 1: Setup & Data

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from src import (
    load_train_speeds, load_train_texts,
    load_adjacency, load_roads,
    build_windows, compute_norm_stats, normalize, denormalize,
    train_val_split, compute_mse, write_submission,
    cache_embeddings, build_entity_masks,
)
from src.model_v3 import SOTATrafficGNNv3
from src.train import (
    build_adj_tensor, select_device,
    encode_texts_qwen, load_qwen,
    train_one_config, build_multi_window_texts,
)

DEVICE = select_device(gpu_id=0)

In [ ]:
s1, s2 = load_train_speeds()
t1_texts, t2_texts = load_train_texts()
adj_raw = load_adjacency()
roads = load_roads()

X1, T1, Y1 = build_windows(s1, t1_texts)
X2, T2, Y2 = build_windows(s2, t2_texts)

X_all = np.concatenate([X1, X2], axis=0).astype(np.float32)
Y_all = np.concatenate([Y1, Y2], axis=0).astype(np.float32)
T_all = np.array(T1 + T2)
T_timestep = np.concatenate([np.arange(len(s1) - 30), np.arange(len(s2) - 30)])

mean, std = compute_norm_stats(np.concatenate([s1, s2], axis=0))
X_norm = normalize(X_all, mean, std)
Y_norm = normalize(Y_all, mean, std)

X_tr, _, Y_tr, X_va, _, Y_va = train_val_split(X_norm, None, Y_norm, val_frac=0.2)
T_tr = T_all[:len(X_tr)]
T_va = T_all[len(X_tr):]
T_ts_tr = T_timestep[:len(X_tr)]
T_ts_va = T_timestep[len(X_tr):]

adj_t = build_adj_tensor(adj_raw).to(DEVICE)

print(f"Train: {X_tr.shape}, Val: {X_va.shape}")

In [ ]:
roadclass = np.array([roads[r][0].get("roadclass", 0) for r in range(1260)], dtype=np.float32)
length = np.array([roads[r][0].get("length", 0) for r in range(1260)], dtype=np.float32)

roadclass_norm = (roadclass - roadclass.mean()) / (roadclass.std() + 1e-8)
length_norm = (length - length.mean()) / (length.std() + 1e-8)

road_meta = torch.tensor(
    np.stack([roadclass_norm, length_norm], axis=-1), dtype=torch.float32
).to(DEVICE)  # (1260, 2)
print(f"Road metadata: {road_meta.shape}")

## Section 2: Text Encoding

Two modes:
- **Single-window**: text at timestep t only (v2-compatible)
- **Multi-window**: texts at t, t-5, t-10 (v3 only, captures event onset)

In [ ]:
qwen_model, qwen_tokenizer = load_qwen("Qwen/Qwen3.5-0.8B", device=DEVICE)

def encode_qwen_wrapper(texts, batch_size=32):
    return encode_texts_qwen(texts, qwen_model, qwen_tokenizer, DEVICE,
                             batch_size=batch_size)

# Single-window (v2 compat)
T_emb_tr, T_emb_va = cache_embeddings(
    T_tr.tolist(), T_va.tolist(),
    encoder_fn=encode_qwen_wrapper,
    cache_name="qwen_single",
    encoder_kwargs={"batch_size": 32},
)
print(f"Single-window: train={T_emb_tr.shape}, val={T_emb_va.shape}")

# Multi-window (v3)
all_texts = t1_texts + t2_texts
T_tr_multi = build_multi_window_texts(all_texts, T_ts_tr, stride=5)
T_va_multi = build_multi_window_texts(all_texts, T_ts_va, stride=5)

T_emb_tr_mw, T_emb_va_mw = cache_embeddings(
    T_tr_multi, T_va_multi,
    encoder_fn=encode_qwen_wrapper,
    cache_name="qwen_multi",
    encoder_kwargs={"batch_size": 32},
)
print(f"Multi-window: train={T_emb_tr_mw.shape}, val={T_emb_va_mw.shape}")

## Section 3: Entity Masks

In [ ]:
entity_masks_tr = build_entity_masks(T_tr.tolist())
entity_masks_va = build_entity_masks(T_va.tolist())

hit_rate = np.mean([m.sum() > 0 for m in entity_masks_tr])
avg_hits = np.mean([m.sum() for m in entity_masks_tr])
print(f"Train entity masks: {hit_rate*100:.1f}% of windows have road mentions, avg {avg_hits:.1f} roads/window")

In [ ]:
class TrafficDS(Dataset):
    def __init__(self, X, T_emb, Y, masks=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.T = torch.tensor(T_emb, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
        self.masks = masks

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        if self.masks is not None:
            return self.X[i], self.T[i], self.Y[i], torch.tensor(self.masks[i])
        return self.X[i], self.T[i], self.Y[i]


def make_loaders(T_emb_tr, T_emb_va, masks_tr=None, masks_va=None, bs=32):
    tr = TrafficDS(X_tr, T_emb_tr, Y_tr, masks_tr)
    va = TrafficDS(X_va, T_emb_va, Y_va, masks_va)
    return (
        DataLoader(tr, batch_size=bs, shuffle=True, pin_memory=True),
        DataLoader(va, batch_size=bs, shuffle=False, pin_memory=True),
    )

print("DataLoaders ready.")

## Section 4: Ablation

| # | Name | Text Adj | Entity Mask | Multi-Window | Metadata | Weighted MSE |
|---|---|---|---|---|---|---|
| A | V2 baseline | — | — | — | — | — |
| B | + TextCondAdj | ✓ | — | — | — | — |
| C | + EntityMask | ✓ | ✓ | — | — | — |
| D | + MultiWindow | ✓ | ✓ | ✓ | — | — |
| E | + Metadata | ✓ | ✓ | ✓ | ✓ | — |
| F | + WeightedMSE | ✓ | ✓ | ✓ | ✓ | ✓ |

In [ ]:
@torch.no_grad()
def eval_mse(model, loader, adj, device):
    model.eval()
    preds, targets = [], []
    for batch in loader:
        if len(batch) == 4:
            Xb, Tb, Yb, Mb = batch
            Mb = Mb.to(device)
        else:
            Xb, Tb, Yb = batch
            Mb = None
        Xb, Tb, Yb = Xb.to(device), Tb.to(device), Yb.to(device)
        pb = model(Xb, Tb, adj, entity_mask=Mb[0] if Mb is not None else None, road_meta=road_meta if road_meta is not None else None)
        preds.append(pb.cpu().numpy())
        targets.append(Yb.cpu().numpy())
    yp = denormalize(np.concatenate(preds, axis=0), mean, std)
    yt = denormalize(np.concatenate(targets, axis=0), mean, std)
    mse = compute_mse(yp, yt)
    for hi, hn in enumerate(["h5", "h10", "h15"]):
        mse_h = compute_mse(yp[:, hi:hi+1, :], yt[:, hi:hi+1, :])
        print(f"  {hn}: {mse_h:.4f}")
    return mse

print("Eval helper ready.")

In [ ]:
import json, pickle, time
from pathlib import Path

ABLATION_DIR = Path("../submissions/ablations")
ABLATION_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FILE = ABLATION_DIR / "results.json"

def save_result(name, model, mse, n_params, n_epochs):
    entry = {"name": name, "mse": round(mse, 4), "params": n_params, "epochs": n_epochs, "time": time.strftime("%Y-%m-%d %H:%M:%S")}
    # Save model
    out = ABLATION_DIR / name
    out.mkdir(exist_ok=True)
    with open(out / "model.pkl", "wb") as f:
        pickle.dump({k: v.cpu() for k, v in model.state_dict().items()}, f)
    with open(out / "result.json", "w") as f:
        json.dump(entry, f, indent=2)
    # Append to results.json (accumulate)
    results = {}
    if RESULTS_FILE.exists():
        results = json.loads(RESULTS_FILE.read_text())
    results[name] = entry
    RESULTS_FILE.write_text(json.dumps(results, indent=2))
    print(f"  Saved to {out}/")
    return mse

results = {}
BASE_KWARGS = dict(d_model=64, num_tcn_layers=4, num_gcn_layers=4, num_hops=2,
                   text_dim=1024, n_heads=4, dropout=0.2, meta_dim=0)
TRAIN_KWARGS = dict(epochs=40, lr=1e-3, patience=8, verbose=True)

# --- A: V2 baseline ---
name = "A_V2_baseline"
print(f"{name}: V2 baseline (no text adj, no mask, single-window)...")
m_a = SOTATrafficGNNv3(use_text=True, use_text_adj=False, use_entity_mask=False, **BASE_KWARGS).to(DEVICE)
tl_a, vl_a = make_loaders(T_emb_tr, T_emb_va, bs=16)
_, n_a, ep_a = train_one_config(m_a, tl_a, vl_a, adj_t, DEVICE, **TRAIN_KWARGS)
mse_a = eval_mse(m_a, vl_a, adj_t, DEVICE)
save_result(name, m_a, mse_a, n_a, ep_a)
results[name] = (mse_a, n_a, ep_a)
print()

# --- B: + TextCondAdj ---
name = "B_TextCondAdj"
print(f"{name}: + TextCondAdj...")
m_b = SOTATrafficGNNv3(use_text=True, use_text_adj=True, use_entity_mask=False, **BASE_KWARGS).to(DEVICE)
tl_b, vl_b = make_loaders(T_emb_tr, T_emb_va, bs=16)
_, n_b, ep_b = train_one_config(m_b, tl_b, vl_b, adj_t, DEVICE, **TRAIN_KWARGS)
mse_b = eval_mse(m_b, vl_b, adj_t, DEVICE)
save_result(name, m_b, mse_b, n_b, ep_b)
results[name] = (mse_b, n_b, ep_b)
print()

# --- C: + EntityMask ---
name = "C_EntityMask"
print(f"{name}: + EntityMask...")
m_c = SOTATrafficGNNv3(use_text=True, use_text_adj=True, use_entity_mask=True, **BASE_KWARGS).to(DEVICE)
tl_c, vl_c = make_loaders(T_emb_tr, T_emb_va, entity_masks_tr, entity_masks_va, bs=16)
_, n_c, ep_c = train_one_config(m_c, tl_c, vl_c, adj_t, DEVICE, **TRAIN_KWARGS)
mse_c = eval_mse(m_c, vl_c, adj_t, DEVICE)
save_result(name, m_c, mse_c, n_c, ep_c)
results[name] = (mse_c, n_c, ep_c)
print()

# --- D: + MultiWindow ---
name = "D_MultiWindow"
print(f"{name}: + MultiWindow...")
m_d = SOTATrafficGNNv3(use_text=True, use_text_adj=True, use_entity_mask=True, **BASE_KWARGS).to(DEVICE)
tl_d, vl_d = make_loaders(T_emb_tr_mw, T_emb_va_mw, entity_masks_tr, entity_masks_va, bs=16)
_, n_d, ep_d = train_one_config(m_d, tl_d, vl_d, adj_t, DEVICE, **TRAIN_KWARGS)
mse_d = eval_mse(m_d, vl_d, adj_t, DEVICE)
save_result(name, m_d, mse_d, n_d, ep_d)
results[name] = (mse_d, n_d, ep_d)
print()

# --- E: + Road Metadata ---
name = "E_Meta"
print(f"{name}: + Road Metadata...")
m_e = SOTATrafficGNNv3(
    use_text=True, use_text_adj=True, use_entity_mask=True, meta_dim=2, **BASE_KWARGS,
).to(DEVICE)
tl_e, vl_e = make_loaders(T_emb_tr, T_emb_va, entity_masks_tr, entity_masks_va, bs=16)
_, n_e, ep_e = train_one_config(m_e, tl_e, vl_e, adj_t, DEVICE, **TRAIN_KWARGS)
mse_e = eval_mse(m_e, vl_e, adj_t, DEVICE)
save_result(name, m_e, mse_e, n_e, ep_e)
results[name] = (mse_e, n_e, ep_e)
print()

# --- F: + WeightedMSE ---
name = "F_WeightedMSE"
print(f"{name}: + WeightedMSE (h5x3, h10x2, h15x1)...")
m_f = SOTATrafficGNNv3(
    use_text=True, use_text_adj=True, use_entity_mask=True, meta_dim=2, **BASE_KWARGS,
).to(DEVICE)
tl_f, vl_f = make_loaders(T_emb_tr, T_emb_va, entity_masks_tr, entity_masks_va, bs=16)
_, n_f, ep_f = train_one_config(m_f, tl_f, vl_f, adj_t, DEVICE,
                                 horizon_weights=[3.0, 2.0, 1.0], **TRAIN_KWARGS)
mse_f = eval_mse(m_f, vl_f, adj_t, DEVICE)
save_result(name, m_f, mse_f, n_f, ep_f)
results[name] = (mse_f, n_f, ep_f)
print()

print("=" * 60)
print(f"{"Config":<25} {"Val MSE":>10} {"Params":>12} {"Epochs":>8}")
print("-" * 57)
for name, (mse, n, ep) in results.items():
    print(f"{name:<25} {mse:>10.4f} {n:>12,} {ep:>8}")

best = min(results, key=lambda k: results[k][0])
print(f"\nBest: {best} ({results[best][0]:.4f})")

## Section 5: Retrain Best & Submit

In [ ]:
T_emb_all = encode_texts_qwen(T_all.tolist(), qwen_model, qwen_tokenizer, DEVICE, batch_size=32)
entity_masks_all = build_entity_masks(T_all.tolist())

full_ds = TrafficDS(X_norm, T_emb_all, Y_norm,
                    masks=entity_masks_all if m_f.use_entity_mask else None)
full_loader = DataLoader(full_ds, batch_size=16, shuffle=True)

model_full = SOTATrafficGNNv3(
    use_text=True, use_text_adj=True, use_entity_mask=True, meta_dim=2,
    d_model=64, num_tcn_layers=4, num_gcn_layers=4,
    num_hops=2, text_dim=1024, n_heads=4, dropout=0.2,
).to(DEVICE)

opt = torch.optim.Adam(model_full.parameters(), lr=1e-3)
hw = torch.tensor([3.0, 2.0, 1.0], device=DEVICE)

for epoch in range(1, 81):
    model_full.train()
    total = 0
    for batch in full_loader:
        if len(batch) == 4:
            Xb, Tb, Yb, Mb = batch
            Mb = Mb.to(DEVICE)
        else:
            Xb, Tb, Yb = batch
            Mb = None
        Xb, Tb, Yb = Xb.to(DEVICE), Tb.to(DEVICE), Yb.to(DEVICE)
        opt.zero_grad()
        pred = model_full(Xb, Tb, adj_t, entity_mask=Mb[0] if Mb is not None else None, road_meta=road_meta)
        loss = ((pred - Yb) ** 2 * hw.view(1, 3, 1)).mean()
        loss.backward()
        nn.utils.clip_grad_norm_(model_full.parameters(), 1.0)
        opt.step()
        total += loss.item() * Xb.size(0)
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | loss: {total / len(full_ds):.4f}")

In [ ]:
from src import load_test_data, build_entity_masks

test_hist, test_texts = load_test_data()
test_norm = normalize(test_hist, mean, std)
test_emb = encode_texts_qwen(test_texts, qwen_model, qwen_tokenizer, DEVICE, batch_size=32)
test_masks = build_entity_masks(test_texts)

@torch.no_grad()
def predict_test(model, X, T_emb, masks, device, bs=16):
    model.eval()
    preds = []
    for i in range(0, len(X), bs):
        xb = torch.tensor(X[i:i+bs], dtype=torch.float32, device=device)
        tb = torch.tensor(T_emb[i:i+bs], dtype=torch.float32, device=device)
        mb = torch.tensor(np.stack(masks[i:i+bs]), dtype=torch.bool, device=device)
        pb = model_full(xb, tb, adj_t, entity_mask=mb[0] if len(mb) > 0 else None, road_meta=road_meta).cpu().numpy()
        preds.append(pb)
    yp_norm = np.concatenate(preds, axis=0)
    return denormalize(yp_norm, mean, std).clip(min=0)

test_preds = predict_test(model_full, test_norm, test_emb, test_masks, DEVICE)
write_submission(test_preds, label="gnn_v3", models={"gnn_v3": model_full.state_dict()})